<a href="https://www.kaggle.com/code/jenifer95/vlm-model?scriptVersionId=357189034" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ELCVIA revision — Kaggle run v2 (video-level split: fine-tuning + descriptions)

**Settings in this version:** 2 epochs, every 2nd training frame (fits one 12-h session). **v2 change:** test videos are held out completely (60 of 298 videos, `ftds_video_split.json`); validation loss uses 24 separate training-side videos. Results go to `results_v2/`, so the old frame-split run is not reused.
Add `ftds_video_split.json` and the patched `paper_results_pipeline.py` to your `elcvia-code` dataset. For the first v2 run, attach the output of the previous run (it contains `ftds_frames.tar`, so no videos are downloaded again).

**Settings (right panel):** Accelerator **GPU P100** (or T4 ×2) · Internet **On** · Input: your dataset `elcvia-code` (and, from the 2nd run on, the **output of the previous version** of this notebook).

Run with **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 h; you can close the browser.
Every run continues where the previous one stopped. When everything is done, download `results/` from the Output tab.

In [1]:
import os, glob, time, shutil, sys, json
START = time.time()
WORK = '/kaggle/working'; os.chdir(WORK)
# find each code file anywhere in the inputs (works whether the dataset holds a zip or loose files)
def find_input(name, must_contain=None):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if must_contain:
        hits = [h for h in hits if must_contain in open(h).read()]
    return hits[0] if hits else None
pipe = find_input('paper_results_pipeline.py', 'ft_train_step')        # the newest patched version only
assert pipe, 'new paper_results_pipeline.py not found - upload it to the elcvia-code dataset'
shutil.copy(pipe, WORK); print('pipeline from:', pipe)
for f in ['revision_experiments.py', 'ftds_pairs.json', 'ftds_video_split.json']:
    p = find_input(f)
    assert p, f + ' not found in the inputs'
    shutil.copy(p, WORK); print(f, 'from:', p)

# continue from a previous version's output (attach it as input)
prev = [p for p in glob.glob('/kaggle/input/**/results_v2/finetune_log.json', recursive=True)] + \
       [p for p in glob.glob('/kaggle/input/**/ftds_frames.tar', recursive=True)]
PREV = os.path.dirname(prev[0]).replace('/results_v2', '') if prev else ''
if PREV:
    print('continuing from', PREV)
    if os.path.isdir(f'{PREV}/results_v2'): shutil.copytree(f'{PREV}/results_v2', f'{WORK}/results_v2', dirs_exist_ok=True)
    for f in ['ftds_local.json', 'ftds_frames.tar']:
        if os.path.exists(f'{PREV}/{f}') and not os.path.exists(f'{WORK}/{f}'): shutil.copy(f'{PREV}/{f}', WORK)
if os.path.exists('ftds_frames.tar') and not os.path.isdir('/tmp/ftds_frames'):
    os.system('tar -xf ftds_frames.tar -C /tmp'); print('keyframes restored')
!df -h /tmp | tail -1

pipeline from: /kaggle/input/datasets/jenifer95/crime-dataset-description/paper_results_pipeline.py
revision_experiments.py from: /kaggle/input/notebooks/jenifer95/vlm-model/revision_experiments.py
ftds_pairs.json from: /kaggle/input/notebooks/jenifer95/vlm-model/ftds_pairs.json
ftds_video_split.json from: /kaggle/input/datasets/jenifer95/crime-dataset-description/ftds_video_split.json
continuing from /kaggle/input/notebooks/jenifer95/vlm-model
keyframes restored
overlay         8.0T  7.0T  1.1T  87% /


In [2]:
!pip -q install --prefer-binary "transformers==4.49.0" timm einops pycocoevalcap bert-score
!nvidia-smi --query-gpu=name,memory.total --format=csv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 24.5 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [3]:
sys.path.insert(0, WORK)
import paper_results_pipeline as P
P.CONFIG.update(pairs=f'{WORK}/ftds_pairs.json', frames_dir='/tmp/ftds_frames', ucf_root='/tmp/videos',
                out=f'{WORK}/results_v2', ft_start='sphar', ft_epochs=2,
                split_mode='video', split_file=f'{WORK}/ftds_video_split.json',
                ft_train_step=2,                       # every 2nd training frame (state in paper)
                deadline=START + 11.3 * 3600)          # stop cleanly before Kaggle's 12-hour limit
P.OUT = P.CONFIG['out']; os.makedirs(P.OUT, exist_ok=True)
class A:
    def __init__(self, **k): self.__dict__.update(k)
def refresh():
    if os.path.exists('ftds_local.json'): P.CONFIG['ftds'] = f'{WORK}/ftds_local.json'
    final = os.path.join(P.OUT, 'checkpoints', f"epoch_{P.CONFIG['ft_epochs']}")
    if os.path.isdir(final): P.CONFIG['ft_model'] = P.CONFIG['ft_processor'] = final
refresh(); print('config ready; frames present:', os.path.isdir('/tmp/ftds_frames'))

config ready; frames present: True


In [4]:
# 1) videos + keyframes (only on the first run)
if not os.path.exists('ftds_frames.tar'):
    VID = '/tmp/videos'; os.makedirs(VID, exist_ok=True)
    PARTS = [
     ('https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/ABYl45nk5Wd0ecKkP7VfnmI/Anomaly-Videos-Part-2.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1', ['Fighting']),
     ('https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/AFb9SoyBX4rJ-XrdqZDVA_I/Anomaly-Videos-Part-3.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1', ['RoadAccidents', 'Shooting']),
     ('https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/AAmA_Jr2it3xfS0o1J-WF1Q/Anomaly-Videos-Part-4.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1', ['Shoplifting']),
    ]
    for url, cats in PARTS:
        os.system(f'wget -q -O /tmp/part.zip "{url}"')
        pats = ' '.join(f"'*{c}*'" for c in cats)
        os.system(f'unzip -q -o /tmp/part.zip {pats} -d {VID}; rm -f /tmp/part.zip')
        print(cats, {c: len(glob.glob(f'{VID}/**/{c}*.mp4', recursive=True)) for c in cats})
    P.stage_build_frames(A()); refresh()
    os.system('tar -cf ftds_frames.tar -C /tmp ftds_frames && rm -rf /tmp/videos')
print('split:', P.CONFIG['split_mode'], '| train pairs:', len(P.ftds_train()), '| val pairs:', len(P.ftds_val()),
      '| test pairs:', len(P.ftds_split()))   # expected 11312 / 1621 / 8520

split: video | train pairs: 11312 | val pairs: 1621 | test pairs: 8520


In [5]:
# 2) fine-tuning (continues from the last finished epoch)
P.stage_finetune(A()); refresh()
print(json.load(open(f'{P.OUT}/finetune_log.json')))

config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

split=video: 11312 train pairs, 1621 validation pairs, 8520 test pairs
epoch 1 step 0/5656 loss 5.2447
epoch 1 step 500/5656 loss 2.8126
epoch 1 step 1000/5656 loss 2.9314
epoch 1 step 1500/5656 loss 2.9395
epoch 1 step 2000/5656 loss 2.3946
epoch 1 step 2500/5656 loss 1.8532
epoch 1 step 3000/5656 loss 2.2291
epoch 1 step 3500/5656 loss 1.8414
epoch 1 step 4000/5656 loss 1.8683
epoch 1 step 4500/5656 loss 1.6931
epoch 1 step 5000/5656 loss 3.0037
epoch 1 step 5500/5656 loss 1.8919
{'epoch': 1, 'train_loss': 2.4953349120546093, 'val_loss': 2.607496107740355, 'minutes': 54.5660474340121}
epoch 2 step 0/5656 loss 2.1779
epoch 2 step 500/5656 loss 2.0564
epoch 2 step 1000/5656 loss 1.9417
epoch 2 step 1500/5656 loss 1.8800
epoch 2 step 2000/5656 loss 1.4435
epoch 2 step 2500/5656 loss 1.5201
epoch 2 step 3000/5656 loss 1.6774
epoch 2 step 3500/5656 loss 2.8294
epoch 2 step 4000/5656 loss 1.6185
epoch 2 step 4500/5656 loss 2.1852
epoch 2 step 5000/5656 loss 2.3864
epoch 2 step 5500/5656 lo

In [6]:
# 3) descriptions from every model (continues where it stopped)
log = json.load(open(f'{P.OUT}/finetune_log.json'))
if len(log['epochs']) >= P.CONFIG['ft_epochs']:
    # most important first, so a session that hits the time limit has the main table done
    E = P.CONFIG['ft_epochs']
    models = [f'ftep{E}', 'base', 'blip', 'blip2'] + [f'ftep{e}' for e in range(1, E)]
    P.stage_captions(A(models=models, limit=0))
    last = f"{P.OUT}/captions_ftep{P.CONFIG['ft_epochs']}.json"
    if os.path.exists(last): shutil.copy(last, f'{P.OUT}/captions_ft.json')
else:
    print('fine-tuning not finished yet - run another version to continue')

FTDS test frames: 8520
ftep2 218 0.426s/frame
ftep2 465 0.391s/frame
ftep2 703 0.392s/frame
ftep2 953 0.387s/frame
ftep2 1157 0.378s/frame
ftep2 1404 0.379s/frame
ftep2 1641 0.368s/frame
ftep2 1841 0.358s/frame
ftep2 2542 0.355s/frame
ftep2 2866 0.354s/frame
ftep2 2867 0.354s/frame
ftep2 2868 0.354s/frame
ftep2 2869 0.354s/frame
ftep2 2870 0.354s/frame
ftep2 2871 0.354s/frame
ftep2 3098 0.352s/frame
ftep2 3315 0.349s/frame
ftep2 3553 0.351s/frame
ftep2 3807 0.351s/frame
ftep2 4029 0.351s/frame
ftep2 4030 0.351s/frame
ftep2 4031 0.351s/frame
ftep2 4032 0.351s/frame
ftep2 4033 0.351s/frame
ftep2 4034 0.351s/frame
ftep2 4035 0.351s/frame
ftep2 4036 0.351s/frame
ftep2 4303 0.351s/frame
ftep2 4525 0.352s/frame
ftep2 4526 0.352s/frame
ftep2 4527 0.352s/frame
ftep2 4762 0.356s/frame
ftep2 4985 0.359s/frame
ftep2 5220 0.361s/frame
ftep2 5221 0.361s/frame
ftep2 5222 0.361s/frame
ftep2 5223 0.361s/frame
ftep2 5224 0.361s/frame
ftep2 5225 0.361s/frame
ftep2 5226 0.361s/frame
ftep2 5437 0.364s/fra

config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/10.0G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

blip2 218 0.494s/frame
blip2 465 0.501s/frame
blip2 703 0.483s/frame
blip2 953 0.481s/frame
blip2 1157 0.497s/frame
blip2 1404 0.494s/frame
blip2 1641 0.495s/frame
blip2 1841 0.495s/frame
blip2 2542 0.494s/frame
blip2 2866 0.496s/frame
blip2 2867 0.496s/frame
blip2 2868 0.496s/frame
blip2 2869 0.496s/frame
blip2 2870 0.496s/frame
blip2 2871 0.496s/frame
blip2 3098 0.499s/frame
blip2 3315 0.501s/frame
blip2 3553 0.502s/frame
blip2 3807 0.504s/frame
blip2 4029 0.504s/frame
blip2 4030 0.504s/frame
blip2 4031 0.504s/frame
blip2 4032 0.504s/frame
blip2 4033 0.504s/frame
blip2 4034 0.504s/frame
blip2 4035 0.504s/frame
blip2 4036 0.504s/frame
blip2 4303 0.507s/frame
blip2 4525 0.507s/frame
blip2 4526 0.507s/frame
blip2 4527 0.507s/frame
blip2 4762 0.507s/frame
blip2 4985 0.506s/frame
blip2 5220 0.506s/frame
blip2 5221 0.506s/frame
blip2 5222 0.506s/frame
blip2 5223 0.506s/frame
blip2 5224 0.506s/frame
blip2 5225 0.506s/frame
blip2 5226 0.506s/frame
blip2 5437 0.506s/frame
blip2 5637 0.506s/fr

In [7]:
# 4) status of this run
tot = len(P.ftds_split())
for f in sorted(glob.glob(f'{P.OUT}/captions_*.json')):
    print(os.path.basename(f), len(json.load(open(f))), '/', tot)
print('epochs done:', len(json.load(open(f'{P.OUT}/finetune_log.json'))['epochs']) if os.path.exists(f'{P.OUT}/finetune_log.json') else 0)
print(f'run time: {(time.time()-START)/3600:.1f} h')
# make a small download bundle with everything except the large model checkpoints
os.system(f'cd {WORK} && zip -q -r results_small.zip results_v2 ftds_video_split.json -x "results_v2/checkpoints/*"')
!ls -lh results_small.zip

captions_base.json 8520 / 8520
captions_blip.json 8520 / 8520
captions_blip2.json 8520 / 8520
captions_ft.json 8520 / 8520
captions_ftep1.json 8520 / 8520
captions_ftep2.json 8520 / 8520
epochs done: 2
run time: 6.9 h


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


-rw-r--r-- 1 root root 1.1M Oct 10 22:26 results_small.zip
